## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
# Load the main application training dataset
data_path           = '../data/raw/'
application         = pd.read_csv(data_path + 'application_train.csv')
prev_application    = pd.read_csv(data_path + 'previous_application.csv')
bureau              = pd.read_csv(data_path + 'bureau.csv')
bureau_balance      = pd.read_csv(data_path + 'bureau_balance.csv')
credit_card_balance = pd.read_csv(data_path + 'credit_card_balance.csv')
installments        = pd.read_csv(data_path + 'installments_payments.csv')
POS_CASH_balance    = pd.read_csv(data_path + 'POS_CASH_balance.csv')



In [ ]:
print(f"application shape: {application.shape}")
print(f"prev_application shape: {prev_application.shape}")
print(f"bureau shape: {bureau.shape}")
print(f"bureau_balance shape: {bureau_balance.shape}")
print(f"credit_card_balance : {credit_card_balance.shape}")
print(f"installments shape: {installments.shape}")
print(f"POSH CASH balance shape: {POS_CASH_balance.shape}")

## 3. Feature Engineering
### 3.1. Application table

In [ ]:
#TRANSFORM VARIABLES
def create_logs(data, features):
    for var in features:
        data["LOG_" + str(var)] = np.log(data[var].abs() + 1)
        data.drop(columns = str(var))
    return data

In [ ]:
class DataPreprocessor:
    def __init__(self, data, is_training = True):
        self.data = data
        self.is_training = is_training
        self.imputer = None 
        self.scaler = None
    def processing_application(self):
        application = self.data.copy()    
        #INCOME RATIO
        application["CREDIT_BY_INCOME"]      = application["AMT_CREDIT"]        / application["AMT_INCOME_TOTAL"]
        application["ANNUITY_BY_INCOME"]     = application["AMT_ANNUITY"]       / application["AMT_INCOME_TOTAL"]
        application["GOODS_PRICE_BY_INCOME"] = application["AMT_GOODS_PRICE"]   / application["AMT_INCOME_TOTAL"]
        application["INCOME_PER_PERSON"]     = application["AMT_INCOME_TOTAL"]  / application["CNT_FAM_MEMBERS"]
        #CREDIT 
        application["CREDIT_TO_GOODS_RATIO"] = application["AMT_CREDIT"] / application["AMT_GOODS_PRICE"]
        application['ANNUITY LENGTH']        = application['AMT_CREDIT'] / application['AMT_ANNUITY']
        #FAMILY STATUS
        application["CNT_ADULTS"]            = application["CNT_FAM_MEMBERS"] - application["CNT_CHILDREN"]
        application['CHILDREN_RATIO']        = application['CNT_CHILDREN'] / application['CNT_FAM_MEMBERS']
        #EMPLOYMENT RATE
        application['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
        application["EMPLOYED_TO_BIRTH_RATIO"] = application["DAYS_EMPLOYED"] / application["DAYS_BIRTH"]
        #NUMBER OF DOCUMENTS
        doc_vars = ["FLAG_DOCUMENT_2",  "FLAG_DOCUMENT_3",  "FLAG_DOCUMENT_4",  "FLAG_DOCUMENT_5",  "FLAG_DOCUMENT_6",
                    "FLAG_DOCUMENT_7",  "FLAG_DOCUMENT_8",  "FLAG_DOCUMENT_9",  "FLAG_DOCUMENT_10", "FLAG_DOCUMENT_11",
                    "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_13", "FLAG_DOCUMENT_14", "FLAG_DOCUMENT_15", "FLAG_DOCUMENT_16",
                    "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_18", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"]
        application["NUM_DOCUMENTS"]    = application[doc_vars].sum(axis = 1)

        application["EXT_SOURCE_MEAN"]  = application[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]].mean(axis = 1)
        #CONTACT
        contact_vars = ["FLAG_MOBIL", "FLAG_EMP_PHONE", "FLAG_WORK_PHONE", "FLAG_CONT_MOBILE", "FLAG_PHONE", "FLAG_EMAIL"]
        application["NUM_CONTACT_METHODS"] = application[contact_vars].sum(axis=1)
        #CAR
        application["CAR_AGE_TO_BIRTH_RATIO"] = application["OWN_CAR_AGE"] / (application["DAYS_BIRTH"] / -365.25)

        #CONVERT DAYS
        day_vars = ["DAYS_BIRTH", "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "DAYS_EMPLOYED", "DAYS_LAST_PHONE_CHANGE"]

        application["AGE"]                      = np.round(application["DAYS_BIRTH"] / (-365), 0)
        application["YEARS_EMPLOYED"]           = np.round(application["DAYS_EMPLOYED"] / (-365), 0)
        application["YEARS_REGISTRATION"]       = np.round(application["DAYS_REGISTRATION"] / (-365), 0)
        application["YEARS_ID_PUBLISH"]         = np.round(application["DAYS_ID_PUBLISH"] / (-365), 0)
        application["YEARS_LAST_PHONE_CHANGE"]  = np.round(application["DAYS_LAST_PHONE_CHANGE"] / (-365), 0)
        
        application.loc[application["AGE"] < 0, "AGE"]                                          = np.nan
        application.loc[application["YEARS_EMPLOYED"] < 0, "YEARS_EMPLOYED"]                    = np.nan
        application.loc[application["YEARS_REGISTRATION"] < 0, "YEARS_REGISTRATION"]            = np.nan
        application.loc[application["YEARS_ID_PUBLISH"] < 0, "YEARS_ID_PUBLISH"]                = np.nan
        application.loc[application["YEARS_LAST_PHONE_CHANGE"] < 0, "YEARS_LAST_PHONE_CHANGE"]  = np.nan

        application.drop(columns=day_vars, inplace=True, errors='ignore')

        #COLUMNS TO DROP
        drops = ['APARTMENTS_MEDI', 'BASEMENTAREA_MEDI', 'COMMONAREA_MEDI', 'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 
         'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI', 'LANDAREA_MEDI', 'LIVINGAPARTMENTS_MEDI', 'LIVINGAREA_MEDI',
         'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI','YEARS_BEGINEXPLUATATION_MEDI', 'YEARS_BUILD_MEDI',
         'APARTMENTS_MODE', 'BASEMENTAREA_MODE', 'COMMONAREA_MODE','ELEVATORS_MODE', 'ENTRANCES_MODE', 
         'FLOORSMAX_MODE', 'FLOORSMIN_MODE', 'LANDAREA_MODE', 'LIVINGAPARTMENTS_MODE', 'LIVINGAREA_MODE', 
         'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAREA_MODE', 'TOTALAREA_MODE',  'YEARS_BEGINEXPLUATATION_MODE']
        application = application.drop(columns = drops)

        return application
    def impute_and_scale(self, df):
        num_cols = df.select_dtypes(include=['float64', 'int64']).columns
        
        if self.is_training:
            self.imputer = SimpleImputer(strategy='median')
            self.scaler  = StandardScaler()
            df[num_cols] = self.imputer.fit_transform(df[num_cols])
            df[num_cols] = self.scaler.fit_transform(df[num_cols])
        else:
            df[num_cols] = self.imputer.transform(df[num_cols])
            df[num_cols] = self.scaler.transform(df[num_cols])
            
        return df


In [ ]:
processor = DataPreprocessor(application)
application_processed = processor.processing_application()
print(application_processed.shape)
display(application_processed.head())